In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = Path.cwd()

MODEL_DIR = PROJECT_ROOT / "models"

print("TensorFlow version:", tf.__version__)
print("Project root:", PROJECT_ROOT)
print("Model directory:", MODEL_DIR)

TensorFlow version: 2.21.0
Project root: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection
Model directory: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\models


In [2]:
# Step 223.3 — Locate VAAC-Tiny model

model_path = MODEL_DIR / "vaac_tiny_best.keras"

print("Model path:")
print(model_path)

print("\nExists:", model_path.exists())


Model path:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\models\vaac_tiny_best.keras

Exists: True


In [3]:
# Step 223.4 — Load trained VAAC-Tiny model

model = tf.keras.models.load_model(model_path)

print("Model loaded successfully.")
print("Model name:", model.name)

Model loaded successfully.
Model name: VAAC_Tiny_CNN


In [4]:
# Step 223.5 — Display model architecture

model.summary()

Model: "VAAC_Tiny_CNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ vibration_input     │ (None, 12000, 1)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ initial_conv        │ (None, 12000, 16) │        128 │ vibration_input[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scale_3             │ (None, 12000, 16) │        320 │ initial_conv[0][… │
│ (SeparableConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scale_7             │ (None, 12000, 16) │        384 │ initial_conv[0][… │
│ (SeparableConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scale_15            │ (None, 12000, 16) │        512 │ initial_conv[0][… │
│ (SeparableConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_scale_fusion  │ (None, 12000, 48) │          0 │ scale_3[0][0],    │
│ (Concatenate)       │                   │            │ scale_7[0][0],    │
│                     │                   │            │ scale_15[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_refinement  │ (None, 12000, 32) │      1,712 │ multi_scale_fusi… │
│ (SeparableConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ feature_refineme… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_features      │ (None, 32)        │      1,056 │ global_average_p… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classification_out… │ (None, 4)         │        132 │ dense_features[0… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,734 (49.75 KB)

 Trainable params: 4,244 (16.58 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,490 (33.17 KB)

In [5]:
# Step 223.6 — Parameter Count

trainable_params = np.sum([
    np.prod(v.shape)
    for v in model.trainable_weights
])

non_trainable_params = np.sum([
    np.prod(v.shape)
    for v in model.non_trainable_weights
])

total_params = trainable_params + non_trainable_params

print("VAAC-Tiny Parameter Analysis")
print("=" * 45)

print(f"Trainable parameters     : {trainable_params:,}")
print(f"Non-trainable parameters : {non_trainable_params:,}")
print(f"Total parameters         : {total_params:,}")

VAAC-Tiny Parameter Analysis
Trainable parameters     : 4,244
Non-trainable parameters : 0.0
Total parameters         : 4,244.0


In [6]:
# Step 223.7 — FP32 Weight Memory Estimate

fp32_weight_bytes = total_params * 4
fp32_weight_kb = fp32_weight_bytes / 1024
fp32_weight_mb = fp32_weight_kb / 1024

print("FP32 Weight Memory")
print("=" * 40)

print(f"Bytes : {fp32_weight_bytes:,}")
print(f"KB    : {fp32_weight_kb:.2f}")
print(f"MB    : {fp32_weight_mb:.4f}")

FP32 Weight Memory
Bytes : 16,976.0
KB    : 16.58
MB    : 0.0162


In [7]:
# Step 223.8 — INT8 Weight Memory Estimate

int8_weight_bytes = total_params
int8_weight_kb = int8_weight_bytes / 1024
int8_weight_mb = int8_weight_kb / 1024

print("INT8 Weight Memory Estimate")
print("=" * 40)

print(f"Bytes : {int8_weight_bytes:,}")
print(f"KB    : {int8_weight_kb:.2f}")
print(f"MB    : {int8_weight_mb:.4f}")

INT8 Weight Memory Estimate
Bytes : 4,244.0
KB    : 4.14
MB    : 0.0040


In [8]:
# Step 223.9 — FP32 vs INT8 Comparison

memory_comparison = pd.DataFrame({
    "Format": ["FP32", "INT8"],
    "Bytes_per_parameter": [4, 1],
    "Estimated_weight_bytes": [
        fp32_weight_bytes,
        int8_weight_bytes
    ],
    "Estimated_weight_KB": [
        fp32_weight_kb,
        int8_weight_kb
    ]
})

display(memory_comparison)

,Format,Bytes_per_parameter,Estimated_weight_bytes,Estimated_weight_KB
0,FP32,4,16976.0,16.578125
1,INT8,1,4244.0,4.144531


In [9]:
# Theoretical memory reduction

memory_reduction = (
    1 - int8_weight_bytes / fp32_weight_bytes
) * 100

print(
    f"Theoretical weight-memory reduction: "
    f"{memory_reduction:.2f}%"
)

Theoretical weight-memory reduction: 75.00%


In [10]:
# Step 223.10 — Layer-wise Parameter Analysis

layer_data = []

for layer in model.layers:

    params = layer.count_params()

    layer_data.append({
        "Layer": layer.name,
        "Type": layer.__class__.__name__,
        "Parameters": params,
        "Output_Shape": str(layer.output.shape)
    })

layer_df = pd.DataFrame(layer_data)

display(layer_df)

,Layer,Type,Parameters,Output_Shape
0,vibration_input,InputLayer,0,"(None, 12000, 1)"
1,initial_conv,Conv1D,128,"(None, 12000, 16)"
2,scale_3,SeparableConv1D,320,"(None, 12000, 16)"
3,scale_7,SeparableConv1D,384,"(None, 12000, 16)"
4,scale_15,SeparableConv1D,512,"(None, 12000, 16)"
5,multi_scale_fusion,Concatenate,0,"(None, 12000, 48)"
6,feature_refinement,SeparableConv1D,1712,"(None, 12000, 32)"
7,global_average_pooling,GlobalAveragePooling1D,0,"(None, 32)"
8,dense_features,Dense,1056,"(None, 32)"
9,classification_output,Dense,132,"(None, 4)"


In [11]:
layer_df["Parameter_Percentage"] = (
    layer_df["Parameters"] /
    total_params *
    100
)

display(layer_df)

,Layer,Type,Parameters,Output_Shape,Parameter_Percentage
0,vibration_input,InputLayer,0,"(None, 12000, 1)",0.000000
1,initial_conv,Conv1D,128,"(None, 12000, 16)",3.016023
2,scale_3,SeparableConv1D,320,"(None, 12000, 16)",7.540057
3,scale_7,SeparableConv1D,384,"(None, 12000, 16)",9.048068
4,scale_15,SeparableConv1D,512,"(None, 12000, 16)",12.064090
5,multi_scale_fusion,Concatenate,0,"(None, 12000, 48)",0.000000
6,feature_refinement,SeparableConv1D,1712,"(None, 12000, 32)",40.339303
7,global_average_pooling,GlobalAveragePooling1D,0,"(None, 32)",0.000000
8,dense_features,Dense,1056,"(None, 32)",24.882187
9,classification_output,Dense,132,"(None, 4)",3.110273


In [12]:
# Step 223.11 — Largest Parameter Layers

largest_layers = (
    layer_df
    .sort_values(
        "Parameters",
        ascending=False
    )
    .reset_index(drop=True)
)

display(largest_layers)

,Layer,Type,Parameters,Output_Shape,Parameter_Percentage
0,feature_refinement,SeparableConv1D,1712,"(None, 12000, 32)",40.339303
1,dense_features,Dense,1056,"(None, 32)",24.882187
2,scale_15,SeparableConv1D,512,"(None, 12000, 16)",12.064090
3,scale_7,SeparableConv1D,384,"(None, 12000, 16)",9.048068
4,scale_3,SeparableConv1D,320,"(None, 12000, 16)",7.540057
5,classification_output,Dense,132,"(None, 4)",3.110273
6,initial_conv,Conv1D,128,"(None, 12000, 16)",3.016023
7,vibration_input,InputLayer,0,"(None, 12000, 1)",0.000000
8,multi_scale_fusion,Concatenate,0,"(None, 12000, 48)",0.000000
9,global_average_pooling,GlobalAveragePooling1D,0,"(None, 32)",0.000000


In [13]:
# Step 223.12 — Input Tensor Memory

input_samples = 12000
input_channels = 1

fp32_input_bytes = (
    input_samples *
    input_channels *
    4
)

int8_input_bytes = (
    input_samples *
    input_channels *
    1
)

print("Input Tensor Memory")
print("=" * 40)

print(f"Input shape: ({input_samples}, {input_channels})")

print(f"\nFP32 input:")
print(f"{fp32_input_bytes:,} bytes")
print(f"{fp32_input_bytes / 1024:.2f} KB")

print(f"\nINT8 input:")
print(f"{int8_input_bytes:,} bytes")
print(f"{int8_input_bytes / 1024:.2f} KB")

Input Tensor Memory
Input shape: (12000, 1)

FP32 input:
48,000 bytes
46.88 KB

INT8 input:
12,000 bytes
11.72 KB


In [14]:
# Step 223.13 — Saved Model File Size

model_file_bytes = os.path.getsize(model_path)

model_file_kb = model_file_bytes / 1024
model_file_mb = model_file_kb / 1024

print("Saved Keras Model Size")
print("=" * 40)

print(f"Bytes : {model_file_bytes:,}")
print(f"KB    : {model_file_kb:.2f}")
print(f"MB    : {model_file_mb:.4f}")

Saved Keras Model Size
Bytes : 110,838
KB    : 108.24
MB    : 0.1057


In [15]:
# Step 223.14 — Input / Output Specification

print("Model Input:")
print(model.input_shape)

print("\nModel Output:")
print(model.output_shape)

print("\nNumber of classes:")
print(model.output_shape[-1])

Model Input:
(None, 12000, 1)

Model Output:
(None, 4)

Number of classes:
4


In [16]:
# Step 223.15 — TinyML Optimization Strategy

optimization_table = pd.DataFrame({
    "Component": [
        "Convolution",
        "Multi-scale branches",
        "Global pooling",
        "Dense layers",
        "Model weights",
        "Input representation",
        "Inference pipeline"
    ],
    "Current_Approach": [
        "Conv1D + SeparableConv1D",
        "3, 7 and 15 sample kernels",
        "GlobalAveragePooling1D",
        "Dense(32) + Dense(4)",
        "FP32 during training",
        "12000-sample vibration window",
        "Software prototype"
    ],
    "Optimization_Direction": [
        "Retain lightweight separable convolutions",
        "Evaluate branch/filter reduction if required",
        "Retain GAP to avoid large Flatten layer",
        "Evaluate compact dense representation",
        "INT8 quantization",
        "Evaluate memory-efficient buffering",
        "Measure actual inference resources"
    ]
})

display(optimization_table)

,Component,Current_Approach,Optimization_Direction
0,Convolution,Conv1D + SeparableConv1D,Retain lightweight separable convolutions
1,Multi-scale branches,"3, 7 and 15 sample kernels",Evaluate branch/filter reduction if required
2,Global pooling,GlobalAveragePooling1D,Retain GAP to avoid large Flatten layer
3,Dense layers,Dense(32) + Dense(4),Evaluate compact dense representation
4,Model weights,FP32 during training,INT8 quantization
5,Input representation,12000-sample vibration window,Evaluate memory-efficient buffering
6,Inference pipeline,Software prototype,Measure actual inference resources


In [17]:
# Step 223.17 — Save Optimization Report

optimization_dir = PROJECT_ROOT / "results" / "optimization"

optimization_dir.mkdir(
    parents=True,
    exist_ok=True
)

optimization_report = pd.DataFrame({
    "Metric": [
        "Total Parameters",
        "Trainable Parameters",
        "Non-trainable Parameters",
        "FP32 Weight Bytes",
        "FP32 Weight KB",
        "INT8 Estimated Weight Bytes",
        "INT8 Estimated Weight KB",
        "Theoretical Weight Memory Reduction (%)",
        "Input Samples",
        "Input Channels",
        "FP32 Input Bytes",
        "INT8 Input Bytes",
        "Saved Keras Model Bytes",
        "Saved Keras Model KB"
    ],
    "Value": [
        total_params,
        trainable_params,
        non_trainable_params,
        fp32_weight_bytes,
        fp32_weight_kb,
        int8_weight_bytes,
        int8_weight_kb,
        memory_reduction,
        input_samples,
        input_channels,
        fp32_input_bytes,
        int8_input_bytes,
        model_file_bytes,
        model_file_kb
    ]
})

report_path = optimization_dir / "vaac_tiny_optimization_report.csv"

optimization_report.to_csv(
    report_path,
    index=False
)

print("Optimization report saved:")
print(report_path)

Optimization report saved:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\optimization\vaac_tiny_optimization_report.csv


In [18]:
# Step 223.18 — Save Layer-wise Analysis

layer_path = (
    optimization_dir /
    "vaac_tiny_layer_parameter_analysis.csv"
)

layer_df.to_csv(
    layer_path,
    index=False
)

print("Layer analysis saved:")
print(layer_path)

Layer analysis saved:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\optimization\vaac_tiny_layer_parameter_analysis.csv


In [19]:
# Step 223.19 — Final TinyML Optimization Verification

print("=" * 60)
print("STEP 223 — TINYML OPTIMIZATION")
print("=" * 60)

print(f"Model                    : {model.name}")
print(f"Input shape              : {model.input_shape}")
print(f"Output shape             : {model.output_shape}")

print(f"\nTotal parameters         : {total_params:,}")
print(f"Trainable parameters     : {trainable_params:,}")

print(f"\nFP32 weight memory       : {fp32_weight_kb:.2f} KB")
print(f"INT8 estimated memory    : {int8_weight_kb:.2f} KB")

print(
    f"Theoretical reduction    : "
    f"{memory_reduction:.2f}%"
)

print(f"\nFP32 input memory        : {fp32_input_bytes / 1024:.2f} KB")
print(f"INT8 input memory        : {int8_input_bytes / 1024:.2f} KB")

print(f"\nKeras file size          : {model_file_kb:.2f} KB")

print("\nStep 223 optimization analysis completed.")

STEP 223 — TINYML OPTIMIZATION
Model                    : VAAC_Tiny_CNN
Input shape              : (None, 12000, 1)
Output shape             : (None, 4)

Total parameters         : 4,244.0
Trainable parameters     : 4,244

FP32 weight memory       : 16.58 KB
INT8 estimated memory    : 4.14 KB
Theoretical reduction    : 75.00%

FP32 input memory        : 46.88 KB
INT8 input memory        : 11.72 KB

Keras file size          : 108.24 KB

Step 223 optimization analysis completed.
